# 0: Environment Setup and Testing

This notebook will:
1. Test your GPU availability and VRAM
2. Install and verify all dependencies
3. Download a small test model
4. Verify the training pipeline works

Run this notebook first to ensure your environment is ready for training.

In [ ]:
# Install dependencies (run once)
!pip install -r ../requirements.txt

## 1. GPU Check

Verify your GPU is available and check VRAM.

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    
    # VRAM
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Total VRAM: {total_vram:.2f} GB")
    
    if total_vram < 8:
        print("⚠️  WARNING: Less than 8GB VRAM detected")
        print("   Consider using smaller batch sizes or gradient accumulation")
else:
    print("❌ No GPU detected. Training will be very slow on CPU.")

## 2. Import Project Modules

Test that all project modules can be imported.

In [ ]:
import sys
sys.path.insert(0, '..')

print("Testing imports...")

# Config
from config.base import ModelConfig, TrainingConfig, LoRAConfig
from config.sft import FINANCE_SFT_CONFIG
print("✓ Config modules")

# Models
from src.models import load_model_and_tokenizer
print("✓ Model modules")

# Data
from src.data import AlpacaDataset, FinanceDataset
print("✓ Data modules")

# Training
from src.training import SFTTrainer
print("✓ Training modules")

# Utils
from src.utils import get_vram_usage, set_seed
print("✓ Utility modules")

print("\n✅ All imports successful!")

## 3. Load a Small Test Model

Download and load a tiny model to test the pipeline.

In [ ]:
# Test with a small model (0.5B parameters)
test_config = ModelConfig(
    name="Qwen/Qwen2.5-0.5B-Instruct",
    quantization_bits=4,
    max_length=512,
)

print("Loading test model (Qwen 0.5B with 4-bit quantization)...")
print("This will download ~1GB of data on first run.\n")

model, tokenizer = load_model_and_tokenizer(test_config)

print("\n✅ Model loaded successfully!")

## 4. Test Text Generation

Generate some text to verify the model works.

In [ ]:
# Test generation
prompt = "### Instruction:\nExplain what a stock is.\n\n### Response:\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=200,
        temperature=0.7,
        do_sample=True,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)

## 5. Test Data Loading

Load and inspect a sample dataset.

In [ ]:
# Load a small sample dataset
dataset = AlpacaDataset(
    data_path="yahma/alpaca-cleaned",
    max_samples=100,
)

dataset.load()

print(f"\nDataset size: {len(dataset)}")
print("\nSample data:")
print(dataset[0])

## 6. Memory Estimation

Estimate VRAM usage for different model sizes.

In [ ]:
from src.utils import estimate_model_vram

# Estimate VRAM for common models
models = [
    "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-3B-Instruct",
]

for model_name in models:
    estimate = estimate_model_vram(
        model_name=model_name,
        quantization_bits=4,
        lora_r=16,
        batch_size=1,
        max_length=1024,
    )
    
    print(f"\n{model_name}:")
    print(f"  Total VRAM: {estimate['total_gb']} GB")
    print(f"  Recommended batch size: {estimate['recommended_batch_size']}")

## 7. Remote Connection Check (Optional)

If using a remote Windows machine for training, test the connection.

In [ ]:
from src.utils import check_remote_gpu

remote = check_remote_gpu("windows")

if remote['available']:
    print(f"✅ Connected to remote machine")
    print(f"GPU: {remote['gpu_name']}")
    print(f"Total Memory: {remote['total_memory_gb']} GB")
    print(f"Free Memory: {remote['free_memory_gb']} GB")
else:
    print(f"❌ Could not connect to remote machine")
    print(f"Error: {remote['error']}")

## ✅ Setup Complete!

Your environment is ready for training. 

### Next Steps:

1. **Learn SFT**: Continue to `01_sft_exploration.ipynb`
2. **Train on Finance**: Use `02_finance_domain.ipynb`
3. **Compare Models**: See `04_model_comparison.ipynb`

### Quick Start Commands:

```bash
# Train on local machine
python scripts/train_sft.py --finance-mode

# Train on remote Windows machine
python scripts/train_remote.py --finance-mode

# Merge LoRA adapters
python scripts/merge_lora.py \
    --base-model Qwen/Qwen2.5-1.5B-Instruct \
    --adapter-path ./outputs/sft
```